# Processo CRISP-DM

* https://www.datascienceprocess.com/wp-content/uploads/2020/05/crisp-dm-guide.pdf 
* https://www.datascience-pm.com/crisp-dm-2/

## Fases

<img src="...">

## Tarefas

<img width=700 src="...
UUAFFFFABRRRQB//2Q==">

## Artefatos

<img width=700 src="data:image/jpg;base64,
...
">

# Python libs + helpers

In [ ]:
# ndarray, Series, DataFrame
import numpy  as np
import pandas as pd

# config:
pd.set_option('float_format', '{:.2f}'.format)
pd.set_option('display.min_rows', 5)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 200)

def left_align(styler):
    f = lambda _: "text-align: left"
    return styler.applymap_index(f, axis=1).applymap(f)

In [ ]:
# visualization libs
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
sns.set()

# interactive browser controls
import ipywidgets as widgets

In [ ]:
# filesystem, datetime
from pathlib  import Path
from datetime import datetime

def dtag(name):
    """ Append date tag to name. """
    return f"{name},{datetime.now().strftime('%Y-%m-%d')}"

In [ ]:
assert False

# Fase 2: Entendimento dos Dados 

## Tabela `DATASETS`

In [ ]:
DATASETS = pd.read_csv('datasets.csv')

def dataset_list(author):
    return DATASETS.query(f'author == "{author}"')

def dataset_read(name, i=0, **kwds):
    path, fmt = DATASETS.query(f'name == "{name}"')[['path', 'format']].iloc[i]
    return pd.read_csv(path, **kwds) if fmt == 'csv' else None

## Problemas de Qualidade dos Dados

### Fonte 1: OWID.ORG

1. **Contagens de valores nulos**:
   * Há uma irregularidade de valores nulos entre as várias métricas;
   * A explicação principal é que certas métricas, em certas localidades, só começaram a ser contadas a partir de determinada data; 
   * Nesse caso o uso de valor nulor reflete uma ausência natural de dados.

### Fonte 2: UN.ORG

1. **Codificação dos datasets**:
   * Não é UTF-8 como requerido pelo ambiente de mineração.
   * **Solução**: converter
2. **Primeira linha dos datasests**:
   * A primeira linha de cada dataset NÃO consiste dos nomes das colunas; 
   * ao invés, tem-se um código e o nome da tabela na primeira linha; 
   * o nome das colunas, ocorre na segunda linha.
   * **Solução**:
     * Deve-se desconsiderar a primeira linha.
3. **Formato numérico do campo valor dos datasets**:
   * Formato de números têm vírgula(s) separando os dígitos em blocos de 3; 
   * Tal formato numérico não é lido pelo pandas.
   * **Solução**:
     * Eliminar vígulas do campo valor de todos os datasets.
4. **Estrutura dos datasets**: 
   * As variáveis de interesse não ocorrem como colunas (como esperado), mas como linhas no dataset;
   * para cada área/região/país, há varias linhas no arquivo csv; 
   * cada linha codifica uma dada variável de interesse em um dado ano.
   * **Solução**: 
     * Mudar estrutura dos datasets mapeando as linhas referentes a uma variável de interesse, para coluna(s). 

# Fase 3: Preparação dos Dados


**Objetivo**: Criação do conjunto de dados a ser utilizado na modelagem.

Em outras palavras, seleção, limpeza e transformação de dados brutos iniciais em dados prontos para mineração.

Em geral, são previstas as atividades:
* **Formatar dados**: Modificar sintaticamente dados (ordem de atributos, remoção de vírgulas, etc.)
* **Selecionar dados**: Decidir que dados (tabelas, registros, atributos) utilizar;
* **Limpar dados**: Remover dados de má qualidade, inserir valores default, estimar valores omissos.
* **Construir dados**: Produzir atributos derivados ou novos registros, transformar valores de atributos existentes.
* **Integar dados**:  Combinar múltiplas tabelas ou registros, agregar informação de múltiplos registros ou tabelas.


A common **rule of thumb** is that <span style="color:red">80% of the project is data preparation </span>. 

## Formatação de Dados

### Exemplo: Conversão ISO_8859-1 para UTF-8

In [ ]:
! ls undata/SYB*.csv,*

In [ ]:
# Converter codificação de ISO_8859-1 para UTF-8;
# Problema observado na coleta dos dados (passo 2.1)

command = 'iconv -f ISO_8859-1 -t UTF-8'
arglist = ! ls undata/SYB*.csv,orig

for orig in arglist:
    utf8 = Path(orig).with_suffix('.csv,utf8')
    if not utf8.exists():
        print(command, orig)
        ! {command} "{orig}" > "{utf8}"

In [ ]:
! rm undata/*,utf8

### Exemplo: Reestruturação dos datasets UN.ORG

In [ ]:
dataset_list('UN.ORG')

In [ ]:
filelist = dataset_list("UN.ORG").path
filelist = widgets.Dropdown(options=filelist, description='Datasets:')

@widgets.interact(x=filelist)
def f(x):
    global DATA
    DATA = pd.read_csv(x, dtype={
        'Year': object,
        'Region/Country/Area': object
    })
    display(DATA)
    print('\n\n>>> Tipos de Dados:\n')
    display(DATA.info())
    print('\n\n>>> Resumo Estatístico:\n')
    display(DATA.describe(include=object).T)
    display(DATA.describe(exclude=object).T)


In [ ]:
DATA.Series.value_counts()

In [ ]:
# ajustar nomes das colunas:
DATA.columns = [ 'Code', 'Name', 'Year', 'Series', 'Value', 'Footnotes', 'Source' ]
DATA

In [ ]:
DATA.query("Name == 'Afghanistan'")

In [ ]:
# pivotar coluna 'Series'
data = DATA.pivot(index=['Code', 'Name', 'Year', 'Source'], columns='Series', values='Value')
data = data.reset_index().rename_axis(None, axis=1).sort_values(by=['Name', 'Year'], ignore_index=True)
data

In [ ]:
data.query("Name == 'Afghanistan'")

## Seleção e Filtragem

### Exemplo: 

In [ ]:
DATA = dataset_read('COVID-19')
DATA.info()

In [ ]:
# Seleção de atributos de interesse:
data = DATA[['iso_code', 'continent', 'date', 'new_deaths']]
data

In [ ]:
# Amostra aleatório de 1% dos dados
data = data.sample(frac=0.01)
data

In [ ]:
# Seleção dos dados da África
data.query('continent == "Africa"')